# Constructing master_table.parquet

Ноутбук собирает все финальные датасеты из `runs`, добавляет полную информацию о прогоне из `run_metadata.json` и конкатенирует их в единую таблицу без удаления исходных столбцов.

Дополнительно вычисляется столбец `incentive`: каноническое имя стимула из `incentives.json` (по `incentive_key` в строке и при необходимости по имени папки прогона) либо строка `no_incentive`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

RUNS_DIR = Path.cwd() / 'runs'
OUTPUT_PATH = Path.cwd() / 'master_table.parquet'
INCENTIVES_PATH = Path.cwd() / 'incentives.json'
RUNS_DIR


In [ ]:
def _resolve_final_dataset_path(run_dir: Path, metadata: dict) -> Path | None:
    """Find final dataset for a run, prioritizing metadata path and then local files."""
    candidate_from_metadata = metadata.get('responses_final_path')
    if candidate_from_metadata:
        p = Path(candidate_from_metadata)
        if p.exists():
            return p

    fallback_candidates = [
        run_dir / 'responses_final.parquet',
        run_dir / 'responses_final.csv',
    ]

    for p in fallback_candidates:
        if p.exists():
            return p

    for pattern in ('responses_final*.parquet', 'responses_final*.csv'):
        matches = sorted(run_dir.glob(pattern))
        if matches:
            return matches[0]

    return None


def _read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Unsupported dataset format: {path}')


def _load_valid_incentive_keys(path: Path) -> set[str]:
    if not path.exists():
        raise FileNotFoundError(f'Expected incentives catalog at {path}')
    data = json.loads(path.read_text(encoding='utf-8'))
    return set(data.keys())


def _incentive_key_from_run_dir_name(run_dir_name: str) -> str | None:
    """Parse canonical incentive key from folder name like incentive_H_..._50_reasoning_..."""
    s = str(run_dir_name)
    if not s.startswith('incentive_'):
        return None
    rest = s.removeprefix('incentive_')
    parts = rest.split('_')
    if '50' in parts:
        return '_'.join(parts[: parts.index('50')])
    return None


def assign_incentive_column(df: pd.DataFrame, valid_keys: set[str]) -> pd.DataFrame:
    """Add column `incentive`: key from incentives.json, or the literal no_incentive."""

    def one(row) -> str:
        ik = row.get('incentive_key')
        if pd.notna(ik):
            t = str(ik).strip()
            if t and t.lower() not in {'nan', 'none', 'null'}:
                return t
        rdn = str(row.get('run_dir_name', ''))
        if rdn.startswith('no_incentive'):
            return 'no_incentive'
        if rdn.startswith('incentive_'):
            parsed = _incentive_key_from_run_dir_name(rdn)
            if parsed and parsed in valid_keys:
                return parsed
            if parsed:
                return parsed
        return 'no_incentive'

    out = df.apply(one, axis=1)
    unknown = set(out.unique()) - valid_keys - {'no_incentive'}
    if unknown:
        raise ValueError(f'incentive values not in incentives.json and not no_incentive: {sorted(unknown)}')
    df = df.copy()
    if 'incentive_key' in df.columns:
        pos = df.columns.get_loc('incentive_key') + 1
        df.insert(pos, 'incentive', out.values)
    else:
        df['incentive'] = out.values
    return df


In [ ]:
metadata_paths = sorted(RUNS_DIR.glob('*/run_metadata.json'))
if not metadata_paths:
    raise FileNotFoundError(f'No run_metadata.json files found in {RUNS_DIR}')

frames = []
missing_final_datasets = []

for meta_path in metadata_paths:
    run_dir = meta_path.parent
    with meta_path.open('r', encoding='utf-8') as f:
        metadata = json.load(f)

    dataset_path = _resolve_final_dataset_path(run_dir, metadata)
    if dataset_path is None:
        missing_final_datasets.append(str(run_dir))
        continue

    df = _read_table(dataset_path)

    # Keep all dataset columns unchanged; add run-level context columns next to them.
    df['run_dir_name'] = run_dir.name
    df['run_dir_path'] = str(run_dir)
    df['source_dataset_path'] = str(dataset_path)
    df['source_metadata_path'] = str(meta_path)

    for key, value in metadata.items():
        col = f'run_meta__{key}'
        if isinstance(value, (dict, list)):
            df[col] = json.dumps(value, ensure_ascii=False, sort_keys=True)
        else:
            df[col] = value

    frames.append(df)

if missing_final_datasets:
    print('Warning: final dataset not found for these runs:')
    for item in missing_final_datasets:
        print('-', item)

if not frames:
    raise RuntimeError('No final datasets were found for any run.')

master_df = pd.concat(frames, ignore_index=True, sort=False)

valid_incentive_keys = _load_valid_incentive_keys(INCENTIVES_PATH)
master_df = assign_incentive_column(master_df, valid_incentive_keys)

master_df.to_parquet(OUTPUT_PATH, index=False)

print(f'Master table saved to: {OUTPUT_PATH}')
print(f'Rows: {len(master_df):,}')
print(f'Columns: {master_df.shape[1]:,}')
print(f'Runs included: {master_df["run_dir_name"].nunique()}')


In [ ]:
master_df.head(3)
